In [1]:
print('In the name of Allah')

In the name of Allah


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score,recall_score


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.svm import SVC

## 1. Data Loading and Initial Exploration

We start by loading our raw data. Exploratory Data Analysis (EDA) is crucial because it helps us identify missing values and understand the data types (numerical vs. categorical) we need to handle later.

In [3]:
# Load the dataset from a CSV file into a pandas DataFrame
df = pd.read_csv("titanic_data_updated.csv")

# Display 5 random rows to see what the data looks like
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
608,609,yes,second,"Laroche, Mrs. Joseph (Juliette Marie Louise La...",female,22.0,1,2,SC/Paris 2123,41.5792,NaN,C
562,563,no,second,"Norman, Mr. Robert Douglas",male,28.0,0,0,218629,13.5000,NaN,S
198,199,yes,third,"Madigan, Miss. Margaret ""Maggie""",female,NaN,0,0,370370,7.7500,NaN,Q
846,847,no,third,"Sage, Mr. Douglas Bullen",male,NaN,8,2,CA. 2343,69.5500,NaN,S
552,553,no,third,"O'Brien, Mr. Timothy",male,NaN,0,0,330979,7.8292,NaN,Q


In [4]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


## 2. Feature Engineering

Feature engineering is the process of using domain knowledge to create new variables that help the model learn better. Here, we calculate `Family_Size` (combining siblings/parents) and extract the `Deck` from the `Cabin` number, as the location on the ship likely influenced survival chances.

In [5]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
700,701,yes,first,"Astor, Mrs. John Jacob (Madeleine Talmadge Force)",female,18.0,1,0,PC 17757,227.5250,C62 C64,C,2,C
19,20,yes,third,"Masselmani, Mrs. Fatima",female,NaN,0,0,2649,7.2250,Missing,C,1,M
131,132,no,third,"Coelho, Mr. Domingos Fernandeo",male,20.0,0,0,SOTON/O.Q. 3101307,7.0500,Missing,S,1,M
457,458,yes,first,"Kenyon, Mrs. Frederick R (Marion)",female,NaN,1,0,17464,51.8625,D21,S,2,D
621,622,yes,first,"Kimball, Mr. Edwin Nelson Jr",male,42.0,1,0,11753,52.5542,D19,S,2,D


In [6]:
df['Deck'].value_counts()

,count
Deck,
M,687
C,59
B,47
D,33
E,32
A,15
F,13
G,4
T,1


In [7]:
# Separate the features (X) from the target we want to predict (y)
# We drop 'Survived' from X because that's our answer key
X = df.drop('Survived', axis=1)

# y contains only the survival status
y = df['Survived']

## 3. Data Splitting

To evaluate our model fairly, we split the data into a **Training Set** and a **Test Set**. We use `stratify=y` to ensure that the proportion of survivors is the same in both sets, preventing the model from becoming biased.

In [8]:
# Split data: 80% for training the model and 20% for testing its accuracy
# 'stratify=y' ensures both sets have a similar percentage of survivors
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 4. Outlier Handling

Outliers can disproportionately influence a model's parameters. We use **Z-scores** to remove rows with extreme 'Age' values and **IQR clipping** to cap extreme 'Fare' prices, ensuring our model generalizes well to average passengers.

In [9]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
692,693,third,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,Missing,S,1,M
481,482,second,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,Missing,S,1,M
527,528,first,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,1,C
855,856,third,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M
801,802,second,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
258,259,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M
736,737,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M
462,463,first,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.5000,E63,S,1,E


In [10]:
y_train

,Survived
692,yes
481,no
527,no
855,yes
801,yes
...,...
359,yes
258,yes
736,no
462,no


In [11]:
#age
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Z_score'] = (X_train['Age'] - mean_age) / std_age

musk = (abs(X_train['Z_score']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

# fare

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

IQR = fare_Q3 - fare_Q1

minimum = max(0 , fare_Q1 - 1.5 * IQR)
maximum = fare_Q3 + 1.5 * IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

/tmp/ipykernel_12857/987313337.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)


## 5. Building Preprocessing Pipelines

Instead of cleaning data manually, we use `Pipelines`. This is a professional standard because it prevents **data leakage** (info from the test set leaking into training) and makes our code reproducible.

*   **Numerical Pipeline**: Fills missing values with the mean/median and scales data.
*   **Categorical Pipeline**: Fills missing values with the most frequent category and encodes text into numbers.

In [12]:
# pipeline

# numerical
p1 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('scaler',StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',MinMaxScaler())
    ]
)

In [13]:
# categories goes one after another via stricly following the order of the input
categories = [['third','second','first']]

In [14]:
# pipeline

#  categorical columns

p3 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OrdinalEncoder(categories=categories)),
        ('scaler',MinMaxScaler())
    ]
)

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1',p1,['Age']),
        ('pipeline_2',p2,['Fare','Family_Size']),
        ('pipeline_3',p3,['Embarked','Sex','Deck']),
        ('pipeline_4',p4,['Pclass'])
    ],
    remainder='drop'
)
preprocessor

ColumnTransformer(transformers=[('pipeline_1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline_2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_Size']),
                                ('pipeline_3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline_4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']])),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])

## 6. Target Variable Encoding

Our target column `Survived` contains 'yes' and 'no'. Since mathematical models require numbers, we use `LabelEncoder` to convert these to `1` and `0` respectively.

In [16]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)

## 7. Model Training and Hyperparameter Tuning

We don't just pick a model; we optimize it. We use **GridSearchCV** to test different settings (kernels and C values) for our Support Vector Classifier. This automatically finds the best configuration using Cross-Validation.

In [17]:
SV_model = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', SVC())
    ]
)


In [18]:
from sklearn.model_selection import GridSearchCV

In [28]:
grid_param = [{
    "model__kernel" : ['linear'],
    "model__C" : [0.01, 0.1, 1, 10, 50, 100]
},
  {
      "model__kernel" : ["rbf"],
      "model__C" : [0.01, 0.1, 1, 100],
      "model__gamma" : [0.001, 0.1, 5, 10, 'scale', 'auto']
  },
    {
      "model__kernel" : ['poly'],
      "model__C" : [0.01, 0.1, 1, 100],
      "model__degree" : [2, 3]
    }

]

In [29]:
best_SVC_model = GridSearchCV(
    estimator = SV_model,
    param_grid = grid_param,
    cv = 5
)

best_SVC_model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('pipeline_1',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Age']),
                                                                        ('pipeline_2',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          MinMaxScaler())]),
                                                                         ['Fare',
                                                                          'Family_Size']),
                                                                        ('pipeline_3',
                                                                         Pipeline(steps=[('im...
                                                                                          OrdinalEncoder(categories=[['third',
                                                                                                                      'second',
                                                                                                                      'first']])),
                                                                                         ('scaler',
                                                                                          MinMaxScaler())]),
                                                                         ['Pclass'])])),
                                       ('model', SVC())]),
             param_grid=[{'model__C': [0.01, 0.1, 1, 10, 50, 100],
                          'model__kernel': ['linear']},
                         {'model__C': [0.01, 0.1, 1, 100],
                          'model__gamma': [0.001, 0.1, 5, 10, 'scale', 'auto'],
                          'model__kernel': ['rbf']},
                         {'model__C': [0.01, 0.1, 1, 100],
                          'model__degree': [2, 3], 'model__kernel': ['poly']}])

In [30]:
# Training Accuracy

y_pred_train = best_SVC_model.predict(X_train)

print(accuracy_score(y_train, y_pred_train))

0.8429319371727748


In [31]:
y_pred = best_SVC_model.predict(X_test)
print(accuracy_score(y_test, y_pred))

y_pred = best_SVC_model.predict(X_test)
print(precision_score(y_test, y_pred))

y_pred = best_SVC_model.predict(X_test)
print(recall_score(y_test, y_pred))

0.8044692737430168
0.8148148148148148
0.6376811594202898
